# LTX Bulk Renderer - Kaggle Launcher

This notebook is a thin orchestration layer.

It prepares the Kaggle runtime, resolves the repository source, performs a self-healing repository bootstrap, validates the source root, imports the repository only after validation, executes the staged bootstrap flow, launches the application, monitors execution, and reports final artifacts.


## Launcher Philosophy

- The notebook is a launcher, not the application.
- Rendering, upload, resume, reporting, validation, Google Drive synchronization, queueing, and stitching remain inside the repository.
- The notebook supports GitHub and Kaggle Dataset repository sources.
- The notebook validates the repository before importing repository modules.
- Dependency installation is delegated to repository dependency profiles rather than `requirements.txt`.


In [4]:
import os
import platform
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

WORKING_ROOT = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
DEFAULT_REPO_URL = 'https://github.com/kcblak/LTX-2.3-22B-Bulk-by_blak_latest.git'
DEFAULT_REPO_REF = 'main'
DEFAULT_REPO_DIRNAME = 'ltx-bulk-renderer-repo'
SOURCE_ROOT_MARKERS = ('main.py', 'config', 'orchestration')
CRITICAL_REPOSITORY_FILES = (
    'main.py',
    'bootstrap.py',
    'config/default.yaml',
    'engine/pipeline.py',
    'renderers/factory.py',
    'reports/report_generator.py',
    'drive/gdrive.py',
    'stitching/service.py',
)


def sanitize_launcher_text(value, *, field_name):
    cleaned = value.strip()
    while len(cleaned) >= 2 and cleaned[0] == cleaned[-1] and cleaned[0] in {'"', "'", '`'}:
        cleaned = cleaned[1:-1].strip()
    if not cleaned:
        raise RuntimeError(f'{field_name} must not be empty after sanitization')
    return cleaned


def sanitize_choice(value, *, field_name, allowed_values):
    cleaned = sanitize_launcher_text(value, field_name=field_name).lower()
    if cleaned not in allowed_values:
        raise RuntimeError(f"{field_name} must be one of: {', '.join(sorted(allowed_values))}")
    return cleaned


def configure_cuda_environment():
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,garbage_collection_threshold:0.6')
    os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
    os.environ.setdefault('MALLOC_TRIM_THRESHOLD_', '0')
    os.environ.setdefault('CUDA_LAUNCH_BLOCKING', '0')


def run_command_result(command, cwd=None, timeout=20):
    started = time.monotonic()
    try:
        result = subprocess.run(command, cwd=cwd, capture_output=True, text=True, timeout=timeout)
        return {
            'command': list(command),
            'cwd': str(cwd) if cwd else None,
            'success': result.returncode == 0,
            'exit_code': result.returncode,
            'stdout': result.stdout,
            'stderr': result.stderr,
            'duration_seconds': round(time.monotonic() - started, 2),
            'timed_out': False,
        }
    except subprocess.TimeoutExpired as exc:
        return {
            'command': list(command),
            'cwd': str(cwd) if cwd else None,
            'success': False,
            'exit_code': -1,
            'stdout': exc.stdout or '',
            'stderr': exc.stderr or '',
            'duration_seconds': round(time.monotonic() - started, 2),
            'timed_out': True,
        }


def try_command(command, cwd=None):
    result = run_command_result(command, cwd=cwd, timeout=5)
    return result['success'], (result['stdout'] or result['stderr'] or '').strip()


def first_nonempty_line(text):
    for line in (text or '').splitlines():
        cleaned = line.strip()
        if cleaned:
            return cleaned
    return ''


def parse_default_branch(text):
    for line in (text or '').splitlines():
        cleaned = line.strip()
        if cleaned.startswith('ref: refs/heads/') and cleaned.endswith('HEAD'):
            return cleaned[len('ref: refs/heads/'):].split('\t', 1)[0].strip()
    return ''


def parse_remote_branches(text):
    branches = []
    for line in (text or '').splitlines():
        cleaned = line.strip()
        if not cleaned or '->' in cleaned:
            continue
        if cleaned.startswith('origin/'):
            branches.append(cleaned.split('/', 1)[1])
    return branches


def describe_working_tree(status_output):
    lines = [line for line in (status_output or '').splitlines() if line.strip()]
    if any(line.startswith('##') and 'detached' in line.lower() for line in lines):
        return 'detached'
    if len(lines) <= 1:
        return 'clean'
    return 'dirty'


def path_looks_like_source_tree(path):
    path = Path(path)
    return path.exists() and any((path / marker).exists() for marker in SOURCE_ROOT_MARKERS)


def detect_source_root(repo_root):
    for candidate in (repo_root, repo_root / 'src'):
        if all((candidate / marker).exists() for marker in SOURCE_ROOT_MARKERS):
            return candidate
    raise RuntimeError(f'Unable to detect a valid source root under {repo_root}')


def validate_repository(source_root):
    missing = [relative_path for relative_path in CRITICAL_REPOSITORY_FILES if not (source_root / relative_path).exists()]
    if missing:
        raise RuntimeError('Repository validation failed. Missing critical files: ' + ', '.join(missing))
    return missing


def discover_dataset_repository(input_root):
    candidates = []
    input_root = Path(input_root)
    search_roots = [input_root] + [path for path in input_root.glob('*') if path.is_dir()]
    for base in search_roots:
        for candidate in (base, base / DEFAULT_REPO_DIRNAME):
            if not candidate.exists() or not candidate.is_dir():
                continue
            score = sum(1 for marker in SOURCE_ROOT_MARKERS if (candidate / marker).exists())
            score += sum(1 for relative_path in CRITICAL_REPOSITORY_FILES if (candidate / relative_path).exists())
            if score:
                candidates.append((score, candidate))
    if not candidates:
        return None
    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[0][1].resolve()


def build_repository_report(destination, repo_url, repo_ref, repository_source, update_policy):
    return {
        'destination': str(Path(destination).resolve()),
        'repo_url': repo_url,
        'configured_ref': repo_ref,
        'target_ref': repo_ref,
        'repository_source': repository_source,
        'update_policy': update_policy,
        'repository_state': 'unknown',
        'current_branch': '',
        'default_branch': '',
        'remote_name': 'origin',
        'remote_url': '',
        'remote_reachable': False,
        'working_tree': 'unknown',
        'checkout': 'SKIPPED',
        'update': 'SKIPPED',
        'recovery': 'Not Required',
        'used_local_copy': False,
        'local_copy_reason': '',
        'validation_messages': [],
        'recovery_actions': [],
        'command_results': [],
        'failure_reason': '',
    }


def git_step(report, command, *, cwd=None, timeout=20):
    result = run_command_result(command, cwd=cwd, timeout=timeout)
    report['command_results'].append(result)
    return result


def ensure_repository_checkout(repo_url, repo_ref, destination, *, repository_source='github', update_policy='auto'):
    destination = Path(destination).resolve()
    report = build_repository_report(destination, repo_url, repo_ref, repository_source, update_policy)

    def use_local_copy(reason, state):
        report['used_local_copy'] = True
        report['local_copy_reason'] = reason
        report['repository_state'] = state
        report['validation_messages'].append(reason)
        report['checkout'] = 'PASS'
        report['update'] = 'SKIPPED'
        report['recovery'] = 'Local Source Tree Used'
        return report

    def clone_fresh(reason):
        report['recovery_actions'].append(reason)
        if destination.exists():
            shutil.rmtree(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)
        clone_result = git_step(report, ['git', 'clone', '--depth', '1', '--branch', report['target_ref'], repo_url, str(destination)], timeout=60)
        if not clone_result['success']:
            report['checkout'] = 'FAILED'
            report['update'] = 'FAILED'
            report['failure_reason'] = (
                f"git clone failed (exit_code={clone_result['exit_code']})\n"
                f"stdout:\n{clone_result['stdout'] or '<empty>'}\n"
                f"stderr:\n{clone_result['stderr'] or '<empty>'}"
            )
            return False
        report['repository_state'] = 'cloned'
        report['checkout'] = 'PASS'
        report['update'] = 'PASS'
        report['current_branch'] = report['target_ref']
        report['default_branch'] = report['target_ref']
        report['remote_url'] = repo_url
        report['remote_reachable'] = True
        report['working_tree'] = 'clean'
        return True

    if repository_source == 'dataset':
        if path_looks_like_source_tree(destination):
            return use_local_copy('Using dataset-provided repository source tree without Git operations.', 'dataset_source')
        report['checkout'] = 'FAILED'
        report['failure_reason'] = f'Dataset repository source was selected, but no valid repository tree exists at {destination}.'
        return report

    if update_policy == 'force' and destination.exists():
        report['recovery'] = 'Fresh Clone Requested'
        if not clone_fresh('Force policy requested a fresh clone.'):
            return report
        return report

    if not destination.exists():
        report['recovery'] = 'Clone Missing Repository'
        if not clone_fresh('Repository directory was missing.'):
            return report
        return report

    if not (destination / '.git').exists():
        if path_looks_like_source_tree(destination):
            return use_local_copy('Destination contains a valid source tree without .git metadata; treating it as a dataset/local copy.', 'local_source_tree')
        report['recovery'] = 'Reclone Invalid Directory'
        if not clone_fresh('Destination existed but was not a Git checkout.'):
            return report
        return report

    git_dir_check = git_step(report, ['git', '-C', str(destination), 'rev-parse', '--git-dir'])
    if not git_dir_check['success']:
        report['recovery'] = 'Reclone Corrupted Metadata'
        if not clone_fresh('Git metadata was invalid or corrupted.'):
            return report
        return report

    report['repository_state'] = 'git_checkout'
    branch_result = git_step(report, ['git', '-C', str(destination), 'branch', '--show-current'])
    report['current_branch'] = first_nonempty_line(branch_result['stdout']) if branch_result['success'] else ''
    if not report['current_branch']:
        report['validation_messages'].append('Detached HEAD detected.')

    status_result = git_step(report, ['git', '-C', str(destination), 'status', '--short', '--branch'])
    if status_result['success']:
        report['working_tree'] = describe_working_tree(status_result['stdout'])

    remote_result = git_step(report, ['git', '-C', str(destination), 'remote', 'get-url', 'origin'])
    if remote_result['success']:
        report['remote_url'] = first_nonempty_line(remote_result['stdout'])
    else:
        add_remote = git_step(report, ['git', '-C', str(destination), 'remote', 'add', 'origin', repo_url])
        if add_remote['success']:
            report['remote_url'] = repo_url
            report['recovery_actions'].append('Recreated missing origin remote.')
        else:
            set_remote = git_step(report, ['git', '-C', str(destination), 'remote', 'set-url', 'origin', repo_url])
            if set_remote['success']:
                report['remote_url'] = repo_url
                report['recovery_actions'].append('Repaired origin remote URL.')

    if report['remote_url'] and report['remote_url'] != repo_url:
        set_url_result = git_step(report, ['git', '-C', str(destination), 'remote', 'set-url', 'origin', repo_url])
        if set_url_result['success']:
            report['remote_url'] = repo_url
            report['recovery_actions'].append('Replaced incorrect origin remote URL.')

    ls_remote_result = git_step(report, ['git', '-C', str(destination), 'ls-remote', '--symref', 'origin', 'HEAD'], timeout=15)
    report['remote_reachable'] = ls_remote_result['success']
    if ls_remote_result['success']:
        report['default_branch'] = parse_default_branch(ls_remote_result['stdout']) or repo_ref
    else:
        report['validation_messages'].append('Remote is not reachable; offline recovery mode enabled.')

    if update_policy == 'never':
        return use_local_copy('Repository update policy is set to never; using the local checkout as-is.', 'local_checkout_policy_never')

    if not report['remote_reachable']:
        if path_looks_like_source_tree(destination):
            return use_local_copy('Remote is unreachable, but a valid local repository copy is available.', 'offline_local_checkout')
        report['checkout'] = 'FAILED'
        report['failure_reason'] = 'Remote repository is unreachable and no valid local source tree is available.'
        return report

    remote_branches_result = git_step(report, ['git', '-C', str(destination), 'branch', '-r'])
    remote_branches = parse_remote_branches(remote_branches_result['stdout']) if remote_branches_result['success'] else []
    if remote_branches and repo_ref not in remote_branches:
        fallback_branch = report['default_branch'] or remote_branches[0]
        report['target_ref'] = fallback_branch
        report['validation_messages'].append(f"Configured branch '{repo_ref}' is unavailable; falling back to '{fallback_branch}'.")
        report['recovery_actions'].append(f"Selected fallback branch '{fallback_branch}'.")
    else:
        report['target_ref'] = repo_ref

    fetch_result = git_step(report, ['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', report['target_ref']], timeout=60)
    if not fetch_result['success']:
        if path_looks_like_source_tree(destination):
            return use_local_copy('Fetch failed, but the local source tree is still usable.', 'local_source_after_fetch_failure')
        report['recovery'] = 'Reclone After Fetch Failure'
        if not clone_fresh('Fetch failed; attempting a fresh clone.'):
            return report
        return report

    checkout_result = git_step(report, ['git', '-C', str(destination), 'checkout', '-B', report['target_ref'], f"origin/{report['target_ref']}"], timeout=30)
    if not checkout_result['success']:
        report['recovery'] = 'Reclone After Checkout Failure'
        if not clone_fresh('Checkout failed; attempting a fresh clone.'):
            return report
        return report

    report['repository_state'] = 'healthy'
    report['current_branch'] = report['target_ref']
    report['remote_url'] = repo_url
    report['working_tree'] = 'clean'
    report['checkout'] = 'PASS'
    report['update'] = 'PASS'
    return report


REPOSITORY_SOURCE = sanitize_choice(os.environ.get('LTX_REPOSITORY_SOURCE', 'github'), field_name='LTX_REPOSITORY_SOURCE', allowed_values={'github', 'dataset'})
REPOSITORY_UPDATE_POLICY = sanitize_choice(os.environ.get('LTX_REPOSITORY_UPDATE_POLICY', 'auto'), field_name='LTX_REPOSITORY_UPDATE_POLICY', allowed_values={'auto', 'never', 'force'})
REPO_URL = sanitize_launcher_text(os.environ.get('LTX_REPO_URL', DEFAULT_REPO_URL), field_name='LTX_REPO_URL')
REPO_REF = sanitize_launcher_text(os.environ.get('LTX_REPO_REF', DEFAULT_REPO_REF), field_name='LTX_REPO_REF')
REPO_DIRNAME = sanitize_launcher_text(os.environ.get('LTX_REPO_DIRNAME', DEFAULT_REPO_DIRNAME), field_name='LTX_REPO_DIRNAME')
REPO_PATH_OVERRIDE = os.environ.get('LTX_REPO_PATH', '').strip()

if REPO_PATH_OVERRIDE:
    REPO_ROOT = Path(sanitize_launcher_text(REPO_PATH_OVERRIDE, field_name='LTX_REPO_PATH')).resolve()
elif REPOSITORY_SOURCE == 'dataset':
    REPO_ROOT = discover_dataset_repository(INPUT_ROOT)
    if REPO_ROOT is None:
        raise RuntimeError('LTX_REPOSITORY_SOURCE=dataset was selected, but no repository source tree was discovered under /kaggle/input. Set LTX_REPO_PATH to the dataset repository path.')
else:
    REPO_ROOT = (WORKING_ROOT / REPO_DIRNAME).resolve()

configure_cuda_environment()

git_ok, git_output = try_command(['git', '--version'])
ffmpeg_ok, ffmpeg_output = try_command(['ffmpeg', '-version'])
nvidia_ok, nvidia_output = try_command(['nvidia-smi'])
disk_usage = shutil.disk_usage(WORKING_ROOT)

print('=== Kaggle Runtime Summary ===')
print('timestamp:', datetime.now().isoformat())
print('python_version:', platform.python_version())
print('platform:', platform.platform())
print('is_kaggle:', Path('/kaggle').exists())
print('working_root:', WORKING_ROOT)
print('input_root:', INPUT_ROOT)
print('disk_free_gb:', round(disk_usage.free / (1024**3), 2))
print('git:', git_output.splitlines()[0] if git_ok and git_output else 'Unavailable')
print('ffmpeg:', ffmpeg_output.splitlines()[0] if ffmpeg_ok and ffmpeg_output else 'Unavailable')
print('gpu_active:', nvidia_ok)
if nvidia_ok and nvidia_output:
    print(nvidia_output.splitlines()[0])
    if len(nvidia_output.splitlines()) > 8:
        print(nvidia_output.splitlines()[8])

checkout_status = ensure_repository_checkout(
    REPO_URL,
    REPO_REF,
    REPO_ROOT,
    repository_source=REPOSITORY_SOURCE,
    update_policy=REPOSITORY_UPDATE_POLICY,
)
if checkout_status['checkout'] == 'FAILED':
    raise RuntimeError(checkout_status['failure_reason'] or 'Repository bootstrap failed')

REPO_ROOT = Path(checkout_status['destination'])
SOURCE_ROOT = detect_source_root(REPO_ROOT)
validate_repository(SOURCE_ROOT)
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(SOURCE_ROOT)

print('\n=== Repository Bootstrap ===')
print('repository_source:', checkout_status['repository_source'])
print('update_policy:', checkout_status['update_policy'])
print('repo_url:', checkout_status['repo_url'])
print('configured_ref:', checkout_status['configured_ref'])
print('target_ref:', checkout_status['target_ref'])
print('repo_root:', REPO_ROOT)
print('source_root:', SOURCE_ROOT)
print('repository_state:', checkout_status['repository_state'])
print('current_branch:', checkout_status['current_branch'] or 'detached/offline-local')
print('default_branch:', checkout_status['default_branch'] or 'unknown')
print('remote_url:', checkout_status['remote_url'] or 'n/a')
print('remote_reachable:', 'YES' if checkout_status['remote_reachable'] else 'NO')
print('working_tree:', checkout_status['working_tree'])
print('checkout:', checkout_status['checkout'])
print('update:', checkout_status['update'])
print('recovery:', checkout_status['recovery'])
if checkout_status['validation_messages']:
    print('validation_messages:')
    for message in checkout_status['validation_messages']:
        print(' -', message)
if checkout_status['recovery_actions']:
    print('recovery_actions:')
    for action in checkout_status['recovery_actions']:
        print(' -', action)
print('dependency_installation:', 'deferred to repository dependency profiles')


=== Kaggle Runtime Summary ===
timestamp: 2026-07-09T09:18:57.120711
python_version: 3.12.13
platform: Linux-6.12.90+-x86_64-with-glibc2.35
is_kaggle: True
working_root: /kaggle/working
input_root: /kaggle/input
disk_free_gb: 19.36
git: git version 2.34.1
ffmpeg: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
gpu_active: True
Thu Jul  9 09:18:56 2026       
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |

=== Repository Bootstrap ===
repository_source: github
update_policy: auto
repo_url: https://github.com/kcblak/LTX-2.3-22B-Bulk-by_blak_latest.git
configured_ref: main
target_ref: main
repo_root: /kaggle/working/ltx-bulk-renderer-repo
source_root: /kaggle/working/ltx-bulk-renderer-repo
repository_state: healthy
current_branch: main
default_branch: main
remote_url: https://github.com/kcblak/LTX-2.3-22B-Bulk-by_blak_latest.git
remote_reachable: YES
working_tree: clean
checkout: PASS
update: PASS
recovery: N

## Phase 1 - Repository Dependency Verification

After the repository is cloned and validated, import the launcher helpers and produce the repository-aware dependency report.


In [ ]:
from orchestration.kaggle import KaggleNotebookLauncher, ensure_notebook_dependencies

inspection = ensure_notebook_dependencies()
print('=== Repository Dependency Report ===')
for package in inspection.packages:
    status = 'installed' if package.installed else 'missing'
    version = package.version or 'n/a'
    print(f' - {package.package_name}: {status} ({version})')
print('ffmpeg:', inspection.ffmpeg_version if inspection.ffmpeg_path else 'missing')
print('cuda_available:', inspection.cuda_available)
print('cuda_version:', inspection.cuda_version)
print('gpu_name:', inspection.gpu_name)


## Phase 2 - Runtime Preparation and Preflight

This phase delegates runtime preparation to the repository. It discovers datasets, restores resume state, detects Google Drive credentials, prepares the Wan2GP runtime and model assets when needed, and runs diagnostics plus preflight.


In [ ]:
launcher = KaggleNotebookLauncher(REPO_ROOT, input_root=INPUT_ROOT, working_root=WORKING_ROOT)
bootstrap_state = launcher.execute_bootstrap_flow()
bootstrap_payload = bootstrap_state.to_dict()
bootstrap_report = bootstrap_state.bootstrap_report.to_dict()
print(launcher.render_bootstrap_report())

context_payload = bootstrap_payload.get('context') or {}
discovery_payload = context_payload.get('discovery') or {}
drive_payload = context_payload.get('drive_credentials') or {}
resume_payload = context_payload.get('resume_summary') or {}
launch_validation_report = {
    'repository_bootstrap': checkout_status,
    'bootstrap_report': bootstrap_report,
    'stage_status': bootstrap_report.get('stages', []),
    'dataset_discovery': {
        'jobs_csv_path': discovery_payload.get('jobs_csv_path'),
        'reference_images_dir': discovery_payload.get('reference_images_dir'),
        'project_config_path': discovery_payload.get('project_config_path'),
        'preset_paths': discovery_payload.get('preset_paths', []),
        'referenced_image_count': discovery_payload.get('referenced_image_count', 0),
        'matched_image_count': discovery_payload.get('image_match_count', 0),
        'missing_image_refs': discovery_payload.get('missing_image_refs', []),
    },
    'drive_detection': {
        'mode': drive_payload.get('source', 'unknown'),
        'enabled': drive_payload.get('enabled', False),
        'notes': drive_payload.get('notes', []),
    },
    'resume': resume_payload,
    'runtime_preparation': bootstrap_payload.get('runtime_preparation', {}),
    'preflight': (bootstrap_payload.get('preparation') or {}).get('preflight', {}),
    'ready_to_launch': bootstrap_report.get('ready_to_launch', False),
}

launch_validation_report


## Phase 3 - Execute Repository Application

This phase launches the repository application and updates the live dashboard from repository heartbeat and manifest data until completion.


In [ ]:
result = launcher.execute_pipeline_stage(refresh_seconds=5)
result_payload = result.to_dict()
result_payload


## Phase 4 - Final Artifact Review

Review the final artifact locations and fail explicitly only after the structured diagnostics and reports have been written.


In [ ]:
execution_state = launcher.get_execution_state().to_dict()
artifact_paths = launcher.get_artifact_report()

for name, info in artifact_paths.items():
    print(f"{name}: {info.get('path')} | exists={info.get('exists')}")

if not result.success:
    raise RuntimeError(
        'Launcher completed with failure after producing structured diagnostics. '
        + result.failure.get('summary', 'Pipeline execution failed')
    )

execution_state
